# 02 — TD3 final training: high N

**Scientific role:** train TD3 with the final constrained physical-action
protocol for `N=[96, 128]`, evaluate all eight seeds on the same
locked 1,000-scenario test banks, and export convergence figures.

**Kaggle settings:** Internet ON, accelerator **NVIDIA Tesla T4**, session
limit 12 hours. The notebook is resume-safe and keeps `best.pt` for the
final latency benchmark.

In [ ]:
from __future__ import annotations

import concurrent.futures
import datetime as dt
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Reproducibility controls
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/Juliolayme/STAR_RIS_RSMA_TD3.git"
REPO_REF = "agent/td3-qos-scalability-v2"
REPO_COMMIT = "89c39da461523a7f5911a302cb9415aeaa5824ce"
REPO_DIR = Path("/kaggle/working/STAR_RIS_RSMA_TD3")
WORKING_DIR = Path("/kaggle/working")

# Keep Python output unbuffered so Kaggle logs remain useful during long runs.
os.environ["PYTHONUNBUFFERED"] = "1"

def run_command(
    command: Sequence[str],
    *,
    cwd: Path,
    log_path: Path,
    extra_env: dict[str, str] | None = None,
) -> None:
    """Run one subprocess, stream its output, and persist an exact text log.

    The function raises immediately on non-zero exit status. This prevents a
    partially failed experiment from being silently included in the final
    thesis tables.
    """
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)

    print("$", " ".join(map(str, command)))
    with log_path.open("w", encoding="utf-8") as handle:
        process = subprocess.Popen(
            list(map(str, command)),
            cwd=str(cwd),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            handle.write(line)
        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Command failed with exit code {return_code}. See {log_path}"
        )

def clone_and_install() -> str:
    """Clone the declared branch, install it editable, and return the commit SHA."""
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    run_command(
        ["git", "clone", "--filter=blob:none", "--branch", REPO_REF, REPO_URL, REPO_DIR],
        cwd=WORKING_DIR,
        log_path=WORKING_DIR / "git_clone.log",
    )
    run_command(
        ["git", "checkout", "--detach", REPO_COMMIT],
        cwd=REPO_DIR,
        log_path=WORKING_DIR / "git_checkout.log",
    )
    run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-build-isolation",
            "-e",
            str(REPO_DIR),
            "tabulate",
        ],
        cwd=REPO_DIR,
        log_path=WORKING_DIR / "pip_install.log",
    )
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
    ).strip()
    if commit != REPO_COMMIT:
        raise RuntimeError(f"Repository drift: expected {REPO_COMMIT}, got {commit}")
    print("Repository commit:", commit)
    return commit

def sha256_file(path: Path) -> str:
    """Return the SHA-256 digest of a file without loading it fully into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def create_locked_banks(n_values: Iterable[int], stage_root: Path) -> None:
    """Create the deterministic train/validation/test ScenarioBanks for each N.

    The command and seeds are inherited from the repository's final GitHub
    Actions protocol. Existing complete banks are reused, which makes reruns
    resume-safe after a Kaggle interruption.
    """
    bank_dir = REPO_DIR / "artifacts" / "scenario_banks"
    bank_dir.mkdir(parents=True, exist_ok=True)
    log_dir = stage_root / "logs" / "scenario_banks"

    for n_ris in n_values:
        test_bank = bank_dir / f"N{n_ris}_test.npz"
        if test_bank.exists():
            print(f"Reuse existing ScenarioBank for N={n_ris}: {test_bank}")
            continue

        config = REPO_DIR / "configs" / "v3" / f"constrained_action_n{n_ris}.yaml"
        run_command(
            [
                sys.executable,
                "scripts/create_scenario_banks.py",
                "--config",
                config,
                "--output-dir",
                bank_dir,
                "--train-count",
                "10000",
                "--validation-count",
                "1000",
                "--test-count",
                "1000",
            ],
            cwd=REPO_DIR,
            log_path=log_dir / f"N{n_ris}.log",
        )

def write_stage_manifest(
    *,
    stage_root: Path,
    stage_id: str,
    commit: str,
    n_values: Sequence[int],
    extra: dict[str, object] | None = None,
) -> None:
    """Write machine-readable provenance consumed by notebook 06."""
    import torch

    payload: dict[str, object] = {
        "stage_id": stage_id,
        "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "repository": REPO_URL,
        "repository_ref": REPO_REF,
        "repository_commit": commit,
        "n_values": list(map(int, n_values)),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device": (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
        ),
    }
    if extra:
        payload.update(extra)
    (stage_root / "STAGE_MANIFEST.json").write_text(
        json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8"
    )


In [ ]:
STAGE_ID = "td3_high_n"
N_VALUES = (96, 128)
STAGE_ROOT = Path("/kaggle/working/thesis_td3_high_n")
STAGE_ROOT.mkdir(parents=True, exist_ok=True)

commit = clone_and_install()
create_locked_banks(N_VALUES, STAGE_ROOT)


In [ ]:
import matplotlib.pyplot as plt

SEEDS = tuple(range(8))
MAX_PARALLEL_TRAININGS = 2

def _valid_test_csv(path: Path, expected_rows: int = 1000) -> bool:
    """Return True only when a completed test CSV covers all expected scenarios."""
    if not path.exists():
        return False
    try:
        frame = pd.read_csv(path)
        scenarios = pd.to_numeric(frame["scenario"], errors="raise").astype(int)
        return (
            len(frame) == expected_rows
            and scenarios.nunique() == expected_rows
            and sorted(scenarios.tolist()) == list(range(expected_rows))
        )
    except Exception:
        return False

def train_and_evaluate_one_seed(
    n_ris: int,
    seed: int,
    stage_root: Path,
) -> dict[str, object]:
    """Train one TD3 seed and evaluate its best checkpoint on the locked test bank.

    Every seed owns a separate output directory and log files. The function is
    safe to call concurrently for different seeds because all jobs only read the
    immutable ScenarioBank files.
    """
    config = REPO_DIR / "configs" / "v3" / f"constrained_action_n{n_ris}.yaml"
    bank = REPO_DIR / "artifacts" / "scenario_banks" / f"N{n_ris}_test.npz"
    seed_root = stage_root / "final_td3_v3" / f"N{n_ris}" / f"seed_{seed}"
    train_root = seed_root / "train"
    test_csv = seed_root / "test.csv"
    seed_root.mkdir(parents=True, exist_ok=True)

    if _valid_test_csv(test_csv) and (train_root / "best.pt").exists():
        print(f"Skip completed TD3 N={n_ris}, seed={seed}")
        return {"n_ris": n_ris, "seed": seed, "status": "reused"}

    run_command(
        [
            sys.executable,
            "scripts/run_train_v2.py",
            "--method",
            "td3",
            "--config",
            config,
            "--seed",
            str(seed),
            "--output",
            train_root,
        ],
        cwd=REPO_DIR,
        log_path=seed_root / "train.log",
        extra_env={"CUDA_VISIBLE_DEVICES": "0"},
    )
    run_command(
        [
            sys.executable,
            "scripts/run_evaluate.py",
            "--method",
            "td3",
            "--config",
            config,
            "--checkpoint",
            train_root / "best.pt",
            "--bank",
            bank,
            "--seed",
            str(seed),
            "--output",
            test_csv,
        ],
        cwd=REPO_DIR,
        log_path=seed_root / "evaluate.log",
        extra_env={"CUDA_VISIBLE_DEVICES": "0"},
    )
    run_command(
        [
            sys.executable,
            "scripts/audit_pilot_output.py",
            "--root",
            seed_root,
            "--seed",
            str(seed),
        ],
        cwd=REPO_DIR,
        log_path=seed_root / "audit.log",
    )

    if not _valid_test_csv(test_csv):
        raise RuntimeError(f"Incomplete TD3 test output: {test_csv}")

    # The latest checkpoint is not used by the final report. Removing it saves
    # Kaggle output storage while retaining the scientifically selected best.pt.
    latest = train_root / "latest.pt"
    if latest.exists():
        latest.unlink()

    return {"n_ris": n_ris, "seed": seed, "status": "completed"}

def run_td3_stage(n_values: Sequence[int], stage_root: Path) -> pd.DataFrame:
    """Run all eight seeds with bounded concurrency and fail on any missing job."""
    jobs = [(int(n), int(seed)) for n in n_values for seed in SEEDS]
    rows: list[dict[str, object]] = []

    with concurrent.futures.ThreadPoolExecutor(
        max_workers=MAX_PARALLEL_TRAININGS
    ) as executor:
        future_map = {
            executor.submit(train_and_evaluate_one_seed, n, seed, stage_root): (n, seed)
            for n, seed in jobs
        }
        for future in concurrent.futures.as_completed(future_map):
            n_ris, seed = future_map[future]
            result = future.result()
            rows.append(result)
            print("Finished:", result)

    status = pd.DataFrame(rows).sort_values(["n_ris", "seed"])
    status.to_csv(stage_root / "TD3_STAGE_STATUS.csv", index=False)
    if len(status) != len(jobs):
        raise RuntimeError("Not all TD3 jobs returned a status row")
    return status

def _save_figure(fig: plt.Figure, output_base: Path) -> None:
    """Save one publication figure as both 300-DPI PNG and vector PDF."""
    output_base.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_base.with_suffix(".png"), dpi=300, bbox_inches="tight")
    fig.savefig(output_base.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)

def plot_td3_stage(n_values: Sequence[int], stage_root: Path) -> None:
    """Create stage-level convergence figures from all available seed CSV files."""
    records: list[pd.DataFrame] = []
    validation_records: list[pd.DataFrame] = []

    for n_ris in n_values:
        for seed in SEEDS:
            train_root = (
                stage_root / "final_td3_v3" / f"N{n_ris}" / f"seed_{seed}" / "train"
            )
            training = pd.read_csv(train_root / "training.csv")
            training["n_ris"] = n_ris
            training["seed"] = seed
            records.append(training)

            validation = pd.read_csv(train_root / "validation_summary.csv")
            validation["n_ris"] = n_ris
            validation["seed"] = seed
            validation_records.append(validation)

    training_all = pd.concat(records, ignore_index=True)
    validation_all = pd.concat(validation_records, ignore_index=True)
    training_all.to_csv(stage_root / "TD3_TRAINING_RAW.csv", index=False)
    validation_all.to_csv(stage_root / "TD3_VALIDATION_RAW.csv", index=False)

    figure_dir = stage_root / "figures"
    for metric, ylabel, use_log in [
        ("sum_rate", "Training sum-rate", False),
        ("qos_fraction", "Training QoS fraction", False),
        ("violation", "Training QoS violation", True),
        ("qos_dual", "Adaptive QoS dual multiplier", False),
    ]:
        fig, ax = plt.subplots(figsize=(7.2, 4.6))
        for n_ris in n_values:
            group = training_all[training_all["n_ris"] == n_ris]
            summary = (
                group.groupby("step")[metric]
                .agg(["mean", "std", "count"])
                .reset_index()
            )
            ci = 1.96 * summary["std"].fillna(0.0) / np.sqrt(summary["count"])
            ax.plot(summary["step"], summary["mean"], label=f"N={n_ris}")
            ax.fill_between(
                summary["step"],
                summary["mean"] - ci,
                summary["mean"] + ci,
                alpha=0.18,
            )
        if use_log:
            ax.set_yscale("log")
        ax.set_xlabel("Training step")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.25)
        ax.legend()
        _save_figure(fig, figure_dir / f"td3_{metric}_convergence")

    # Validation curves are the primary evidence for checkpoint selection.
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for n_ris in n_values:
        group = validation_all[validation_all["n_ris"] == n_ris]
        summary = (
            group.groupby("eval_step")["mean_sum_rate"]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        ci = 1.96 * summary["std"].fillna(0.0) / np.sqrt(summary["count"])
        ax.plot(summary["eval_step"], summary["mean"], label=f"N={n_ris}")
        ax.fill_between(
            summary["eval_step"],
            summary["mean"] - ci,
            summary["mean"] + ci,
            alpha=0.18,
        )
    ax.set_xlabel("Training step")
    ax.set_ylabel("Validation sum-rate")
    ax.grid(True, alpha=0.25)
    ax.legend()
    _save_figure(fig, figure_dir / "td3_validation_sum_rate")


## Execute the final eight-seed protocol

Two training subprocesses share the T4 by default. This bounded
concurrency is intended to stay within Kaggle memory while finishing
comfortably inside a 12-hour session. Change `MAX_PARALLEL_TRAININGS`
to `1` only if the particular Kaggle image reports CUDA memory pressure.

In [ ]:
status = run_td3_stage(N_VALUES, STAGE_ROOT)
display(status)


## Export convergence evidence and provenance

In [ ]:
plot_td3_stage(N_VALUES, STAGE_ROOT)
write_stage_manifest(
    stage_root=STAGE_ROOT,
    stage_id=STAGE_ID,
    commit=commit,
    n_values=N_VALUES,
    extra={
        "seeds": list(SEEDS),
        "expected_test_scenarios_per_seed": 1000,
        "training_protocol": "td3_qos_scalability_v3_constrained",
        "action_parameterization": "physical_v3",
        "max_parallel_trainings": MAX_PARALLEL_TRAININGS,
    },
)

print("Stage output:", STAGE_ROOT)
print("Files:", sum(1 for _ in STAGE_ROOT.rglob("*") if _.is_file()))
